Data Preparation & Exploratory Data Analysis


In [2]:
from google.colab import files
uploaded = files.upload()


Saving customer_shopping_behavior.csv to customer_shopping_behavior.csv


In [3]:
import pandas as pd
import numpy as np

# Optional (for later visualization)
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
df = pd.read_csv("customer_shopping_behavior.csv")

df.head()


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases,Frequency Standardized
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,Bi-Weekly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,Bi-Weekly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,Annually


In [5]:
df.shape

(3900, 19)

In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [7]:
df.describe()


,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [10]:
#Missing Review Ratings

df['Review Given'] = df['Review Rating'].apply(lambda x: 'No' if pd.isna(x) else 'Yes')


In [9]:
df['Review Given'].value_counts()


,count
Review Given,
Yes,3863
No,37


In [11]:
# Clean Discount & Promo Columns

print(df['Discount Applied'].unique())
print(df['Promo Code Used'].unique())
print(df['Subscription Status'].unique())


['Yes' 'No']
['Yes' 'No']
['Yes' 'No']


In [12]:
# Convert Yes/No Columns to Binary - easier for statistical testing.

binary_cols = ['Discount Applied', 'Promo Code Used', 'Subscription Status', 'Review Given']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df[binary_cols].head()


,Discount Applied,Promo Code Used,Subscription Status,Review Given
0,1,1,1,1
1,1,1,1,1
2,1,1,1,1
3,1,1,1,1
4,1,1,1,1


In [13]:
# Age Group - This allows:

- Revenue by age segment

- Loyalty by age segment

- Marketing targeting analysis

- Dashboard-friendly segmentation

def age_group(age):
    if age <= 25:
        return "18-25"
    elif age <= 35:
        return "26-35"
    elif age <= 50:
        return "36-50"
    elif age <= 65:
        return "51-65"
    else:
        return "66-70"

df['Age Group'] = df['Age'].apply(age_group)

df['Age Group'].value_counts()


,count
Age Group,
51-65,1121
36-50,1111
26-35,742
18-25,571
66-70,355


In [14]:
# Segment by Previous Purchases

def loyalty_tier(purchases):
    if purchases <= 12:
        return "Low"
    elif purchases <= 25:
        return "Moderate"
    elif purchases <= 38:
        return "High"
    else:
        return "Very High"

df['Loyalty Tier'] = df['Previous Purchases'].apply(loyalty_tier)

df['Loyalty Tier'].value_counts()


,count
Loyalty Tier,
Moderate,1020
High,1007
Low,945
Very High,928


High loyalty tiers generate more total revenue primarily because

- There are more customers in those tiers

- Not because they spend more per purchase

Low loyalty customers actually have slightly higher average purchase amount.


It suggests:

Spending amount per purchase is independent of loyalty depth

In [18]:
# Revenue by Loyalty Tier

df.groupby('Loyalty Tier')['Purchase Amount (USD)'].agg(['count','mean','sum']).sort_values(by='sum', ascending=False)


,count,mean,sum
Loyalty Tier,,,
High,1007,59.885799,60305
Moderate,1020,58.818627,59995
Low,945,60.448677,57124
Very High,928,59.975216,55657


In [22]:
# percentage of subscribers in each tier

pd.crosstab(df['Loyalty Tier'], df['Subscription Status'], normalize='index') * 100

# This tells us if subscription drives loyalty.

Subscription Status,0,1
Loyalty Tier,,
High,70.804369,29.195631
Low,75.026455,24.973545
Moderate,73.333333,26.666667
Very High,72.952586,27.047414


Subscription rate increases slightly as loyalty increases — but not dramatically.

Low tier → 24.9% subscribed
High tier → 29.2% subscribed

Difference ≈ 4%

That’s weak.

In [23]:
# discount usage influences loyalty

pd.crosstab(df['Loyalty Tier'], df['Discount Applied'], normalize='index') * 100



Discount Applied,0,1
Loyalty Tier,,
High,58.490566,41.509434
Low,58.412698,41.587302
Moderate,55.686275,44.313725
Very High,55.387931,44.612069


Discount usage does NOT strongly differentiate loyalty levels.

So:

Subscription → Weak influence

Discount → Weak influence

In [24]:
# does frequency align with loyalty.

pd.crosstab(df['Loyalty Tier'], df['Frequency Standardized'], normalize='index') * 100


Frequency Standardized,Annually,Bi-Weekly,Monthly,Quarterly,Weekly
Loyalty Tier,,,,,
High,14.101291,28.202582,15.491559,27.209533,14.995035
Low,15.132275,29.206349,13.015873,29.523810,13.121693
Moderate,15.980392,27.450980,15.294118,27.941176,13.333333
Very High,13.362069,26.831897,12.715517,33.297414,13.793103


Loyalty Tier is largely independent of stated frequency.

So far:

Subscription → weak differentiator

Discount → weak differentiator

Frequency category → weak differentiator

Purchase amount → almost equal across tiers

So loyalty segmentation is mostly reflecting historical depth (Previous Purchases), not current behavioral signals.

In [25]:
# revenue contribution % per tier.

total_revenue = df['Purchase Amount (USD)'].sum()

(df.groupby('Loyalty Tier')['Purchase Amount (USD)'].sum() / total_revenue) * 100


,Purchase Amount (USD)
Loyalty Tier,
High,25.872980
Low,24.508218
Moderate,25.739979
Very High,23.878823


In [26]:
# Which tier has highest average review rating

df.groupby('Loyalty Tier')['Review Rating'].mean()


,Review Rating
Loyalty Tier,
High,3.740622
Low,3.775880
Moderate,3.721344
Very High,3.765649


Subscription → weak difference

Discount usage → weak difference

Frequency label → weak difference

Review rating → weak difference

Purchase amount → almost equal

This dataset is structurally balanced.

In [27]:
# Which Age Groups drive the most revenue

age_revenue = df.groupby('Age Group')['Purchase Amount (USD)'].agg(['count','mean','sum'])
age_revenue.sort_values(by='sum', ascending=False)


,count,mean,sum
Age Group,,,
51-65,1121,60.281891,67576
36-50,1111,59.072007,65629
26-35,742,59.760108,44342
18-25,571,60.647986,34630
66-70,355,58.884507,20904


The 36–65 age demographic represents the primary revenue engine of the business. Strategic marketing efforts and loyalty programs should prioritize retention and personalization for this segment

In [28]:
# revenue contribution percentage by age group.

total_revenue = df['Purchase Amount (USD)'].sum()

(df.groupby('Age Group')['Purchase Amount (USD)'].sum() / total_revenue) * 100


,Purchase Amount (USD)
Age Group,
18-25,14.857496
26-35,19.024288
36-50,28.157164
51-65,28.992496
66-70,8.968556


Nearly 57% of total revenue comes from customers aged 36–65.

That is a dominant demographic cluster.

Marketing investment and retention initiatives should prioritize this segment to protect core revenue streams.

In [30]:
#Average Previous Purchases by Age Group
df.groupby('Age Group')['Previous Purchases'].mean()


,Previous Purchases
Age Group,
18-25,24.633975
26-35,24.760108
36-50,24.896490
51-65,26.307761
66-70,26.146479


Revenue difference is mainly due to customer volume (count), not loyalty depth.

In [31]:
# Does Subscription Status impact spending?

df.groupby('Subscription Status')['Purchase Amount (USD)'].agg(['count','mean','sum'])


,count,mean,sum
Subscription Status,,,
0,2847,59.865121,170436
1,1053,59.491928,62645


Average spend per purchase is almost identical

59.87 vs 59.49
→ Difference is negligible.

Non-subscribers generate more total revenue
BUT that’s because there are more of them (2847 vs 1053).


In [32]:
# if subscribers purchase more often historically.

df.groupby('Subscription Status')['Previous Purchases'].mean()



,Previous Purchases
Subscription Status,
0,25.080436
1,26.084520


Identify True Revenue Drivers

In [33]:
# category-level revenue.

df.groupby('Category')['Purchase Amount (USD)'].agg(['count','mean','sum']).sort_values(by='sum', ascending=False)


,count,mean,sum
Category,,,
Clothing,1737,60.025331,104264
Accessories,1240,59.838710,74200
Footwear,599,60.255426,36093
Outerwear,324,57.172840,18524


Clothing is the primary revenue engine, contributing the largest share of total revenue due to high transaction volume rather than premium pricing.

In [34]:
# category has the highest customer satisfaction?

df.groupby('Category')['Review Rating'].mean().sort_values(ascending=False)


,Review Rating
Category,
Footwear,3.793771
Accessories,3.769976
Outerwear,3.745652
Clothing,3.721537


Clothing drives the most revenue but has the lowest customer satisfaction among categories.

Do Clothing customers use discounts more?

In [35]:
pd.crosstab(df['Category'], df['Discount Applied'], normalize='index') * 100


Discount Applied,0,1
Category,,
Accessories,56.209677,43.790323
Clothing,57.915947,42.084053
Footwear,56.761269,43.238731
Outerwear,55.555556,44.444444


Discount usage is almost identical across all categories.

So:

Clothing’s lower satisfaction is NOT driven by heavy discounting.
Footwear’s higher rating is NOT due to fewer discounts.

In [36]:
df.groupby('Category')['Purchase Amount (USD)'].mean()


,Purchase Amount (USD)
Category,
Accessories,59.838710
Clothing,60.025331
Footwear,60.255426
Outerwear,57.172840


In [37]:
# % distribution of customers per category.

df['Category'].value_counts(normalize=True) * 100


,proportion
Category,
Clothing,44.538462
Accessories,31.794872
Footwear,15.358974
Outerwear,8.307692


In [38]:
# Revenue by Gender

df.groupby('Gender')['Purchase Amount (USD)'].agg(['count','mean','sum'])


,count,mean,sum
Gender,,,
Female,1248,60.249199,75191
Male,2652,59.536199,157890


In [40]:
# Revenue Contribution %

total_revenue = df['Purchase Amount (USD)'].sum()
(df.groupby('Gender')['Purchase Amount (USD)'].sum() / total_revenue) * 100


,Purchase Amount (USD)
Gender,
Female,32.259601
Male,67.740399


Female customers are not less valuable per transaction.

There are simply fewer female customers in this dataset.

In [42]:
# Does Gender Preference Differ by Category?
pd.crosstab(df['Category'], df['Gender'], normalize='index') * 100



Gender,Female,Male
Category,,
Accessories,31.612903,68.387097
Clothing,32.009211,67.990789
Footwear,33.222037,66.777963
Outerwear,31.172840,68.827160


The business is structurally male-dominated across the board.
Female underrepresentation is consistent, not category-specific.

## Revenue Risk **Concentration**

In [43]:
risk_segment = df[
    (df['Category'] == 'Clothing') &
    (df['Age Group'].isin(['36-50','51-65'])) &
    (df['Gender'] == 'Male')
]

risk_segment['Purchase Amount (USD)'].sum() / df['Purchase Amount (USD)'].sum() * 100


np.float64(16.70234811074262)

Clothing is the dominant category.

36–65 is the dominant age group.

Males dominate customer base.

Combined, this cluster forms ~17% of total revenue.

Clothing also has lowest satisfaction rating.

**Final Insights & Strategic Summary**

1️⃣ Revenue Structure Overview

The dataset contains 3,900 customers with evenly distributed revenue across loyalty tiers. No extreme Pareto concentration was observed. Revenue is broadly balanced across customer segments, indicating a structurally stable retail model.

2️⃣ Category-Level Intelligence

Clothing accounts for ~45% of customers and is the largest revenue contributor.

However, Clothing has the lowest average customer satisfaction rating among all categories.

Revenue dominance is driven by high transaction volume rather than higher pricing.

Implication: Clothing represents both the primary revenue engine and a potential satisfaction risk area.

3️⃣ Growth Opportunity Segment

Footwear shows the highest customer satisfaction rating.

Average purchase value is comparable to other categories.

However, transaction volume is significantly lower (~15% customer share).

Implication: Strategic marketing and assortment expansion in Footwear may unlock incremental revenue growth.

4️⃣ Demographic Intelligence

Customers aged 36–65 contribute ~57% of total revenue, making this the primary revenue-driving demographic.

Revenue is heavily male-dominated (~68%), although average spending per transaction is nearly identical between genders.

Implication: Expanding female customer acquisition represents a scalable revenue opportunity.

5️⃣ Subscription & Discount Impact

Subscription status does not significantly impact per-transaction spending or historical purchase depth.

Discount usage is evenly distributed across segments and categories.

Implication: Current subscription and discount programs may not be strong revenue or loyalty drivers.

6️⃣ Revenue Concentration Risk

Approximately 16.7% of total revenue originates from middle-aged male customers purchasing Clothing.

Implication: While overall revenue is balanced, certain demographic-category clusters represent moderate concentration risk if satisfaction declines.